# 02. Praproses dan Pipeline

**Tujuan:** menyiapkan praproses yang dipakai bersama oleh semua model, tanpa kebocoran data.

**Rancangan** (kode ada di `src/pipeline.py`):
- Membersihkan nama kolom.
- *One-hot encoding* untuk kolom kategorik berkode angka (program studi, jalur pendaftaran, pendidikan dan pekerjaan orang tua, dan lainnya).
- Penskalaan standar untuk kolom numerik; kolom biner dibiarkan.
- Semuanya berada di dalam `Pipeline`, sehingga hanya di-*fit* pada data latih di setiap putaran validasi silang.

**Dua skenario fitur:** *tanpa akademik* (24 fitur, informasi saat pendaftaran) dan *dengan akademik* (36 fitur, ditambah kinerja semester 1 dan 2).

In [2]:
import sys
sys.path.append("..")

from src.pipeline import muat_data, pilih_skenario, buat_preprocessor

X_train, y_train = muat_data("../data/processed/train.csv")
print(X_train.shape, y_train.mean().round(3))

for dengan in (False, True):
    Xs = pilih_skenario(X_train, dengan)
    Z = buat_preprocessor(Xs).fit_transform(Xs)
    print("dengan akademik:", dengan, "| fitur mentah:", Xs.shape[1], "| setelah praproses:", Z.shape[1])

(3539, 36) 0.321
dengan akademik: False | fitur mentah: 24 | setelah praproses: 223
dengan akademik: True | fitur mentah: 36 | setelah praproses: 235


In [2]:
import sys
sys.path.append("..")

from src.pipeline import muat_data, pilih_skenario, KATEGORIK, BINER

X_train, y_train = muat_data("../data/processed/train.csv")
Xs = pilih_skenario(X_train, True)
kat = [c for c in KATEGORIK if c in Xs.columns]
biner = [c for c in BINER if c in Xs.columns]
numerik = [c for c in Xs.columns if c not in kat + biner]

print(f"Kategorik ({len(kat)} kolom): di-one-hot")
print(f"Biner ({len(biner)} kolom): dibiarkan")
print(f"Numerik ({len(numerik)} kolom): distandarkan")
print("Nilai hilang:", int(X_train.isna().sum().sum()), "| Baris duplikat:", int(X_train.duplicated().sum()))

Kategorik (9 kolom): di-one-hot
Biner (8 kolom): dibiarkan
Numerik (19 kolom): distandarkan
Nilai hilang: 0 | Baris duplikat: 0


In [3]:
from src.pipeline import buat_preprocessor

pre = buat_preprocessor(Xs).fit(Xs)
nama_kolom = pre.get_feature_names_out()
print(len(nama_kolom), "kolom setelah praproses")
print("Awal :", list(nama_kolom[:3]))
print("Akhir:", list(nama_kolom[-3:]))

235 kolom setelah praproses
Awal : ['kat__Marital status_1', 'kat__Marital status_2', 'kat__Marital status_3']
Akhir: ['biner__Gender', 'biner__Scholarship holder', 'biner__International']


## Kesimpulan

- Skenario tanpa akademik: 24 fitur mentah menjadi 223 kolom setelah praproses. Skenario dengan akademik: 36 fitur mentah menjadi 235 kolom.
- Jumlah kolom membengkak karena kolom kategorik berkode angka (misalnya program studi dan pekerjaan orang tua) memiliki banyak kategori.
- Praproses ini dipakai oleh semua model pada notebook 03.